# Acquisition des données play-by-play de la LNH

Ce notebook présente la méthode utilisée pour récupérer les données
play-by-play de la LNH pour les saisons 2016-2017 à 2023-2024.

L'objectif est de construire un outil réutilisable permettant :

- de récupérer les identifiants des matchs d'une saison ;
- de télécharger les données play-by-play depuis l'API de la LNH ;
- de sauvegarder les données brutes localement ;
- d'utiliser une copie locale lorsqu'un match a déjà été téléchargé ;
- de séparer les données par saison et par type de match
  (saison régulière et séries éliminatoires).

Les fonctions principales sont implémentées dans le module
`ift6758.data_acquisition`.

## 1. Importation des fonctions d'acquisition

La logique principale d'acquisition est placée dans un module Python
réutilisable plutôt que directement dans le notebook.

In [ ]:
from pathlib import Path

from ift6758.data_acquisition import (
    build_regular_game_id,
    download_game,
    download_season,
    get_game_ids,
)

## 2. Structure du GAME_ID

Chaque match de la LNH possède un identifiant unique (`GAME_ID`).

Pour un match de saison régulière, l'identifiant suit la structure :

`YYYY02NNNN`

où :

- `YYYY` correspond à l'année de début de la saison ;
- `02` correspond à un match de saison régulière ;
- `NNNN` correspond au numéro du match.

Par exemple, le premier match de saison régulière de la saison
2023-2024 possède l'identifiant `2023020001`.

In [74]:
game_id = build_regular_game_id(2023, 1)

print("GAME_ID :", game_id)

GAME_ID : 2023020001


## 3. Récupération des GAME_ID d'une saison

Plutôt que de générer tous les identifiants possibles, nous interrogeons
l'API de la LNH afin d'obtenir uniquement les matchs réellement associés
à une saison et à un type de match donné.

La fonction `get_game_ids` accepte :

- l'identifiant de la saison, par exemple `20232024` ;
- le type de match :
  - `2` pour la saison régulière ;
  - `3` pour les séries éliminatoires.

Cela permet d'éviter des requêtes inutiles vers des GAME_ID inexistants.

In [75]:
regular_ids = get_game_ids(20232024, 2)
playoff_ids = get_game_ids(20232024, 3)

print("Saison régulière :", len(regular_ids))
print("Premier GAME_ID :", regular_ids[0])
print("Dernier GAME_ID :", regular_ids[-1])

print()

print("Séries éliminatoires :", len(playoff_ids))
print("Premier GAME_ID :", playoff_ids[0])
print("Dernier GAME_ID :", playoff_ids[-1])

Saison régulière : 1312
Premier GAME_ID : 2023020001
Dernier GAME_ID : 2023021312

Séries éliminatoires : 105
Premier GAME_ID : 2023030111
Dernier GAME_ID : 2023030417


## 4. Téléchargement d'un match et mise en cache locale

La fonction `download_game` récupère les données play-by-play d'un match
à partir de son `GAME_ID`.

Le comportement est le suivant :

1. si le fichier JSON existe déjà localement, les données sont chargées
   depuis le cache ;
2. sinon, une requête est envoyée à l'API de la LNH ;
3. la réponse JSON est sauvegardée localement ;
4. les données sont retournées sous forme de dictionnaire Python.

Cette stratégie permet d'éviter de télécharger plusieurs fois les mêmes
données et réduit le nombre de requêtes envoyées à l'API publique.

In [77]:
game = download_game(
    2023020001,
    data_dir=Path("../data/raw/20232024/regular")
)

print("Game ID :", game["id"])
print("Date :", game["gameDate"])
print("Nombre d'événements :", len(game["plays"]))

Chargement depuis le cache : 2023020001.json
Game ID : 2023020001
Date : 2023-10-10
Nombre d'événements : 328


Le message `Chargement depuis le cache` confirme qu'aucune nouvelle
requête HTTP n'a été envoyée lorsque les données étaient déjà disponibles
localement.

## 5. Téléchargement d'une saison complète

La fonction `download_season` combine la récupération des GAME_ID
et le téléchargement des données play-by-play.

Pour chaque saison, les matchs sont séparés en deux catégories :

- `regular` pour la saison régulière ;
- `playoffs` pour les séries éliminatoires.

Les fichiers JSON sont sauvegardés selon la structure suivante :

`data/raw/<saison>/<type_de_match>/`

Par exemple :

`data/raw/20232024/regular/`

et

`data/raw/20232024/playoffs/`

La fonction accepte également le paramètre optionnel `max_games`, utile
pour effectuer de petits tests avant de lancer un téléchargement complet.

In [78]:
# Exemple de test avec deux matchs par type

download_season(20232024, max_games=2)


Téléchargement de 2 matchs (regular) pour la saison 20232024
Chargement depuis le cache : 2023020001.json
Chargement depuis le cache : 2023020002.json

Téléchargement de 2 matchs (playoffs) pour la saison 20232024
Chargement depuis le cache : 2023030111.json
Chargement depuis le cache : 2023030112.json


Pour télécharger une saison complète, il suffit d'omettre le paramètre
`max_games` :

```python
download_season(20232024)

## 6. Validation des données téléchargées

Après le téléchargement, nous vérifions que les fichiers disponibles
localement correspondent aux matchs retournés par l'API.

Pour chaque saison, nous comparons :

- le nombre de GAME_ID retournés par l'API ;
- le nombre de fichiers JSON sauvegardés localement.

Pour les séries éliminatoires, certains GAME_ID retournés par la liste
des matchs ne possèdent pas de données play-by-play disponibles.
Dans ce cas, l'endpoint play-by-play retourne une erreur HTTP 404.
Ces matchs sont ignorés par l'outil plutôt que de créer des fichiers
JSON artificiels.

In [79]:
seasons = [
    20162017,
    20172018,
    20182019,
    20192020,
    20202021,
    20212022,
    20222023,
    20232024,
]

data_root = Path("../data/raw")

for season_id in seasons:
    expected_regular = len(get_game_ids(season_id, 2))
    expected_playoffs = len(get_game_ids(season_id, 3))

    regular_dir = data_root / str(season_id) / "regular"
    playoffs_dir = data_root / str(season_id) / "playoffs"

    local_regular = len(list(regular_dir.glob("*.json")))
    local_playoffs = len(list(playoffs_dir.glob("*.json")))

    print(f"Saison {season_id}")
    print(f"  Regular  : {local_regular}/{expected_regular}")
    print(
        f"  Playoffs : {local_playoffs}/{expected_playoffs} "
        f"(sans PBP : {expected_playoffs - local_playoffs})"
    )
    print()

Saison 20162017
  Regular  : 1230/1230
  Playoffs : 87/87 (sans PBP : 0)

Saison 20172018
  Regular  : 1271/1271
  Playoffs : 84/84 (sans PBP : 0)

Saison 20182019
  Regular  : 1271/1271
  Playoffs : 87/87 (sans PBP : 0)

Saison 20192020
  Regular  : 1082/1082
  Playoffs : 130/134 (sans PBP : 4)

Saison 20202021
  Regular  : 868/868
  Playoffs : 84/105 (sans PBP : 21)

Saison 20212022
  Regular  : 1312/1312
  Playoffs : 89/105 (sans PBP : 16)

Saison 20222023
  Regular  : 1312/1312
  Playoffs : 88/105 (sans PBP : 17)

Saison 20232024
  Regular  : 1312/1312
  Playoffs : 88/105 (sans PBP : 17)



Les saisons régulières sont entièrement disponibles localement.

Pour certaines saisons, le nombre de fichiers de séries éliminatoires
est inférieur au nombre de GAME_ID retournés par l'API. Cette différence
correspond aux GAME_ID pour lesquels l'endpoint play-by-play retourne
une réponse HTTP 404.

L'outil traite ces cas comme des données play-by-play indisponibles et
poursuit le téléchargement des autres matchs.

## 7. Résumé

L'outil d'acquisition permet donc :

- d'obtenir les GAME_ID d'une saison ;
- de télécharger les données play-by-play d'un match ;
- de mettre automatiquement les réponses en cache localement ;
- de télécharger une saison complète ;
- de séparer les données par saison régulière et séries éliminatoires ;
- de continuer le téléchargement lorsqu'un GAME_ID ne possède pas de
  données play-by-play accessibles.

Les données brutes des saisons 2016-2017 à 2023-2024 ont été récupérées
et sont conservées localement dans `data/raw/`.

Le dossier `data/` est exclu du dépôt Git afin d'éviter de versionner
un grand nombre de fichiers de données brutes.

In [3]:
# Importer la bibliothèque requests pour communiquer avec l'API de la LNH
import requests

# Identifiant d'un match connu de la saison 2023-2024
game_id = 2023021190

# Construire l'adresse de l'API play-by-play
url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"

# Envoyer une requête HTTP à l'API
response = requests.get(url)

# Afficher le code de statut HTTP
print("Code de statut :", response.status_code)

Code de statut : 200


In [4]:
# Convertir la réponse JSON en dictionnaire Python
game_data = response.json()

# Vérifier le type de l'objet obtenu
print(type(game_data))

# Afficher les principales clés disponibles
print(game_data.keys())

<class 'dict'>
dict_keys(['id', 'season', 'gameType', 'limitedScoring', 'gameDate', 'venue', 'venueLocation', 'startTimeUTC', 'easternUTCOffset', 'venueUTCOffset', 'tvBroadcasts', 'gameState', 'gameScheduleState', 'periodDescriptor', 'awayTeam', 'homeTeam', 'shootoutInUse', 'otInUse', 'clock', 'displayPeriod', 'maxPeriods', 'gameOutcome', 'plays', 'rosterSpots', 'regPeriods', 'summary'])


In [5]:
# Afficher quelques informations générales sur le match

print("Game ID :", game_data["id"])
print("Saison :", game_data["season"])
print("Type de match :", game_data["gameType"])
print("Date :", game_data["gameDate"])
print("Équipe visiteuse :", game_data["awayTeam"]["abbrev"])
print("Équipe locale :", game_data["homeTeam"]["abbrev"])

Game ID : 2023021190
Saison : 20232024
Type de match : 2
Date : 2024-04-02
Équipe visiteuse : FLA
Équipe locale : MTL


In [6]:
# Récupérer la liste des événements du match
plays = game_data["plays"]

# Vérifier le type de la variable
print("Type :", type(plays))

# Afficher le nombre total d'événements
print("Nombre total d'événements :", len(plays))

Type : <class 'list'>
Nombre total d'événements : 345


In [7]:
premier_evenement = plays[0]
premier_evenement

{'eventId': 52,
 'periodDescriptor': {'number': 1,
  'periodType': 'REG',
  'maxRegulationPeriods': 3},
 'timeInPeriod': '00:00',
 'timeRemaining': '20:00',
 'situationCode': '1551',
 'homeTeamDefendingSide': 'right',
 'typeCode': 520,
 'typeDescKey': 'period-start',
 'sortOrder': 8}

In [8]:
# Chercher le premier tir cadré dans la liste des événements
premier_tir = None

for evenement in plays:
    if evenement["typeDescKey"] == "shot-on-goal":
        premier_tir = evenement
        break

# Afficher le premier tir trouvé
premier_tir

{'eventId': 108,
 'periodDescriptor': {'number': 1,
  'periodType': 'REG',
  'maxRegulationPeriods': 3},
 'timeInPeriod': '00:28',
 'timeRemaining': '19:32',
 'situationCode': '1551',
 'homeTeamDefendingSide': 'right',
 'typeCode': 506,
 'typeDescKey': 'shot-on-goal',
 'sortOrder': 18,
 'details': {'xCoord': 53,
  'yCoord': 36,
  'zoneCode': 'O',
  'shotType': 'wrist',
  'shootingPlayerId': 8475765,
  'goalieInNetId': 8478470,
  'eventOwnerTeamId': 13,
  'awaySOG': 1,
  'homeSOG': 0}}

In [9]:
# Récupérer les détails du premier tir
details_tir = premier_tir["details"]

# Afficher les détails
details_tir

{'xCoord': 53,
 'yCoord': 36,
 'zoneCode': 'O',
 'shotType': 'wrist',
 'shootingPlayerId': 8475765,
 'goalieInNetId': 8478470,
 'eventOwnerTeamId': 13,
 'awaySOG': 1,
 'homeSOG': 0}

In [10]:
# Afficher quelques informations importantes sur le tir

print("Coordonnée X :", details_tir.get("xCoord"))
print("Coordonnée Y :", details_tir.get("yCoord"))
print("Type de tir :", details_tir.get("shotType"))
print("ID du tireur :", details_tir.get("shootingPlayerId"))
print("ID du gardien :", details_tir.get("goalieInNetId"))
print("ID de l'équipe :", details_tir.get("eventOwnerTeamId"))

Coordonnée X : 53
Coordonnée Y : 36
Type de tir : wrist
ID du tireur : 8475765
ID du gardien : 8478470
ID de l'équipe : 13


In [11]:
# Chercher le premier but dans la liste des événements
premier_but = None

for evenement in plays:
    if evenement["typeDescKey"] == "goal":
        premier_but = evenement
        break

# Afficher le premier but trouvé
premier_but

{'eventId': 235,
 'periodDescriptor': {'number': 1,
  'periodType': 'REG',
  'maxRegulationPeriods': 3},
 'timeInPeriod': '09:16',
 'timeRemaining': '10:44',
 'situationCode': '1551',
 'homeTeamDefendingSide': 'right',
 'typeCode': 505,
 'typeDescKey': 'goal',
 'sortOrder': 110,
 'details': {'xCoord': 84,
  'yCoord': 6,
  'zoneCode': 'O',
  'shotType': 'snap',
  'scoringPlayerId': 8482113,
  'scoringPlayerTotal': 10,
  'eventOwnerTeamId': 13,
  'goalieInNetId': 8478470,
  'awayScore': 1,
  'homeScore': 0,
  'highlightClipSharingUrl': 'https://nhl.com/video/fla-mtl-lundell-scores-goal-against-samuel-montembeault-6350163860112',
  'highlightClipSharingUrlFr': 'https://nhl.com/fr/video/fla-mtl-lundell-scores-goal-against-samuel-montembeault-6350164935112',
  'highlightClip': 6350163860112,
  'highlightClipFr': 6350164935112,
  'discreteClip': 6350164450112,
  'discreteClipFr': 6350165037112},
 'pptReplayUrl': 'https://wsr.nhle.com/sprites/20232024/2023021190/ev235.json'}

In [12]:
# Récupérer les détails du premier but
details_but = premier_but["details"]

# Afficher les détails du but
details_but

{'xCoord': 84,
 'yCoord': 6,
 'zoneCode': 'O',
 'shotType': 'snap',
 'scoringPlayerId': 8482113,
 'scoringPlayerTotal': 10,
 'eventOwnerTeamId': 13,
 'goalieInNetId': 8478470,
 'awayScore': 1,
 'homeScore': 0,
 'highlightClipSharingUrl': 'https://nhl.com/video/fla-mtl-lundell-scores-goal-against-samuel-montembeault-6350163860112',
 'highlightClipSharingUrlFr': 'https://nhl.com/fr/video/fla-mtl-lundell-scores-goal-against-samuel-montembeault-6350164935112',
 'highlightClip': 6350163860112,
 'highlightClipFr': 6350164935112,
 'discreteClip': 6350164450112,
 'discreteClipFr': 6350165037112}

In [13]:
# Afficher quelques informations importantes sur le but

print("Coordonnée X :", details_but.get("xCoord"))
print("Coordonnée Y :", details_but.get("yCoord"))
print("Type de tir :", details_but.get("shotType"))

print("ID du buteur :", details_but.get("scoringPlayerId"))
print("ID du gardien :", details_but.get("goalieInNetId"))
print("ID de l'équipe :", details_but.get("eventOwnerTeamId"))

print("Score visiteur :", details_but.get("awayScore"))
print("Score local :", details_but.get("homeScore"))

# Certains buts peuvent contenir cette information
print("Force :", details_but.get("strength"))

Coordonnée X : 84
Coordonnée Y : 6
Type de tir : snap
ID du buteur : 8482113
ID du gardien : 8478470
ID de l'équipe : 13
Score visiteur : 1
Score local : 0
Force : None


In [14]:
def download_game(game_id):
    """
    Télécharger les données play-by-play d'un match de la LNH.

    Parameters
    ----------
    game_id : int
        Identifiant unique du match.

    Returns
    -------
    dict
        Données JSON du match sous forme de dictionnaire Python.
    """

    # Construire l'URL du match
    url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"

    # Envoyer la requête à l'API
    response = requests.get(url, timeout=30)

    # Vérifier si la requête a réussi
    response.raise_for_status()

    # Convertir la réponse JSON en dictionnaire Python
    return response.json()

In [15]:
# Tester la fonction avec un match connu
test_game = download_game(2023021190)

print("Game ID :", test_game["id"])
print("Date :", test_game["gameDate"])
print("Nombre d'événements :", len(test_game["plays"]))

Game ID : 2023021190
Date : 2024-04-02
Nombre d'événements : 345


In [16]:
from pathlib import Path

# Afficher le dossier de travail actuel
print(Path.cwd())

C:\Users\17960\Desktop\IFT3700\Projet1\notebooks


In [17]:
import json

# Définir le dossier où sauvegarder les données brutes
data_dir = Path("..") / "data" / "raw"

# Créer le dossier s'il n'existe pas encore
data_dir.mkdir(parents=True, exist_ok=True)

# Définir le chemin du fichier JSON
file_path = data_dir / "2023021190.json"

# Sauvegarder les données du match dans un fichier JSON
with open(file_path, "w", encoding="utf-8") as fichier:
    json.dump(test_game, fichier, ensure_ascii=False, indent=2)

print("Fichier sauvegardé :", file_path.resolve())

Fichier sauvegardé : C:\Users\17960\Desktop\IFT3700\Projet1\data\raw\2023021190.json


In [18]:
# Charger les données sauvegardées depuis le fichier JSON
with open(file_path, "r", encoding="utf-8") as fichier:
    donnees_locales = json.load(fichier)

# Vérifier que les données ont bien été chargées
print("Game ID :", donnees_locales["id"])
print("Date :", donnees_locales["gameDate"])
print("Nombre d'événements :", len(donnees_locales["plays"]))

Game ID : 2023021190
Date : 2024-04-02
Nombre d'événements : 345


In [19]:
def download_game(game_id, data_dir=Path("..") / "data" / "raw"):
    """
    Télécharger ou charger les données play-by-play d'un match de la LNH.

    Parameters
    ----------
    game_id : int
        Identifiant unique du match.

    data_dir : Path
        Dossier où les fichiers JSON sont sauvegardés.

    Returns
    -------
    dict
        Données du match sous forme de dictionnaire Python.
    """

    # Créer le dossier de données s'il n'existe pas
    data_dir.mkdir(parents=True, exist_ok=True)

    # Construire le chemin du fichier local
    file_path = data_dir / f"{game_id}.json"

    # Si le fichier existe déjà, charger les données locales
    if file_path.exists():
        print("Chargement depuis le cache :", file_path.name)

        with open(file_path, "r", encoding="utf-8") as fichier:
            return json.load(fichier)

    # Sinon, télécharger les données depuis l'API
    print("Téléchargement depuis l'API :", game_id)

    url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    game_data = response.json()

    # Sauvegarder les données localement
    with open(file_path, "w", encoding="utf-8") as fichier:
        json.dump(game_data, fichier, ensure_ascii=False, indent=2)

    return game_data

In [20]:
# Tester le chargement depuis le cache
test_cache = download_game(2023021190)

print("Game ID :", test_cache["id"])
print("Nombre d'événements :", len(test_cache["plays"]))

Chargement depuis le cache : 2023021190.json
Game ID : 2023021190
Nombre d'événements : 345


In [23]:
# Tester le téléchargement depuis l'API avec un autre match
nouveau_match = download_game(2023020001)

print("Game ID :", nouveau_match["id"])
print("Date :", nouveau_match["gameDate"])
print("Nombre d'événements :", len(nouveau_match["plays"]))

Chargement depuis le cache : 2023020001.json
Game ID : 2023020001
Date : 2023-10-10
Nombre d'événements : 328


In [54]:
# Importer la fonction depuis le module du projet
from ift6758.data_acquisition import download_game, build_regular_game_id, get_game_ids, download_season

In [10]:
# Tester la fonction importée avec un match déjà en cache
game_test = download_game(2023021190)

print("Game ID :", game_test["id"])
print("Date :", game_test["gameDate"])
print("Nombre d'événements :", len(game_test["plays"]))

Chargement depuis le cache : 2023021190.json
Game ID : 2023021190
Date : 2024-04-02
Nombre d'événements : 345


In [11]:
# Tester la construction de plusieurs GAME_ID

print(build_regular_game_id(2023, 1))
print(build_regular_game_id(2023, 25))
print(build_regular_game_id(2023, 1190))

2023020001
2023020025
2023021190


In [53]:
# Recharger le module après sa modification

import importlib
import ift6758.data_acquisition as data_acquisition

importlib.reload(data_acquisition)

<module 'ift6758.data_acquisition' from 'C:\\Users\\17960\\Desktop\\IFT3700\\Projet1\\ift6758\\data_acquisition.py'>

In [13]:
# Récupérer une petite liste de matchs d'une saison
# Importer requests pour communiquer avec l'API
import requests

season_id = 20232024

url_games = "https://api.nhle.com/stats/rest/en/game"

params = {
    "cayenneExp": f"season={season_id}",
    "limit": 5
}

response_games = requests.get(url_games, params=params, timeout=30)
response_games.raise_for_status()

games_data = response_games.json()

# Afficher les clés principales de la réponse
print(games_data.keys())

# Afficher le nombre total de matchs trouvés
print("Nombre total :", games_data.get("total"))

dict_keys(['data', 'total'])
Nombre total : 1532


In [14]:
# Afficher le premier match retourné par l'API

games_data.get("data", [])[0]

{'id': 2023010001,
 'easternStartTime': '2023-09-23T00:05:00',
 'gameDate': '2023-09-23',
 'gameNumber': 1,
 'gameScheduleStateId': 1,
 'gameStateId': 6,
 'gameType': 1,
 'homeScore': 5,
 'homeTeamId': 53,
 'period': 3,
 'season': 20232024,
 'visitingScore': 3,
 'visitingTeamId': 26}

In [15]:
# Récupérer les matchs de saison régulière

season_id = 20232024
game_type = 2

params_regular = {
    "cayenneExp": f"season={season_id} and gameType={game_type}",
    "limit": 5
}

response_regular = requests.get(
    url_games,
    params=params_regular,
    timeout=30
)

response_regular.raise_for_status()

regular_data = response_regular.json()

print("Nombre de matchs de saison régulière :", regular_data["total"])

Nombre de matchs de saison régulière : 1312


In [16]:
# Afficher le premier match de saison régulière
premier_match_regulier = regular_data["data"][0]

premier_match_regulier

{'id': 2023020001,
 'easternStartTime': '2023-10-10T17:30:00',
 'gameDate': '2023-10-10',
 'gameNumber': 1,
 'gameScheduleStateId': 1,
 'gameStateId': 7,
 'gameType': 2,
 'homeScore': 5,
 'homeTeamId': 14,
 'period': 3,
 'season': 20232024,
 'visitingScore': 3,
 'visitingTeamId': 18}

In [17]:
# Récupérer les matchs des séries éliminatoires

season_id = 20232024
game_type = 3

params_playoffs = {
    "cayenneExp": f"season={season_id} and gameType={game_type}",
    "limit": 5
}

response_playoffs = requests.get(
    url_games,
    params=params_playoffs,
    timeout=30
)

response_playoffs.raise_for_status()

playoffs_data = response_playoffs.json()

print(
    "Nombre de matchs des séries éliminatoires :",
    playoffs_data["total"]
)

Nombre de matchs des séries éliminatoires : 105


In [18]:
# Afficher le premier match des séries éliminatoires

premier_match_playoffs = playoffs_data["data"][0]

premier_match_playoffs

{'id': 2023030111,
 'easternStartTime': '2024-04-21T12:30:00',
 'gameDate': '2024-04-21',
 'gameNumber': 111,
 'gameScheduleStateId': 1,
 'gameStateId': 7,
 'gameType': 3,
 'homeScore': 3,
 'homeTeamId': 13,
 'period': 3,
 'season': 20232024,
 'visitingScore': 2,
 'visitingTeamId': 14}

In [33]:
# Récupérer les GAME_ID de la saison 2023-2024

regular_ids = get_game_ids(20232024, 2)
playoff_ids = get_game_ids(20232024, 3)

print("Saison régulière :", len(regular_ids))
print("Premier GAME_ID :", regular_ids[0])
print("Dernier GAME_ID :", regular_ids[-1])

print()

print("Séries éliminatoires :", len(playoff_ids))
print("Premier GAME_ID :", playoff_ids[0])
print("Dernier GAME_ID :", playoff_ids[-1])

Saison régulière : 1312
Premier GAME_ID : 2023020001
Dernier GAME_ID : 2023021312

Séries éliminatoires : 105
Premier GAME_ID : 2023030111
Dernier GAME_ID : 2023030417


In [46]:
# Vérifier que les données de la saison 2016-2017 sont accessibles

regular_ids_2016 = get_game_ids(20162017, 2)
playoff_ids_2016 = get_game_ids(20162017, 3)

print("Saison régulière :", len(regular_ids_2016))
print("Premier GAME_ID :", regular_ids_2016[0])

print()

print("Séries éliminatoires :", len(playoff_ids_2016))
print("Premier GAME_ID :", playoff_ids_2016[0])

Saison régulière : 1230
Premier GAME_ID : 2016020001

Séries éliminatoires : 87
Premier GAME_ID : 2016030111


In [ ]:
# Vérifier le nombre attendu de matchs pour la saison 2017-2018

regular_ids_2017 = get_game_ids(20172018, 2)
playoff_ids_2017 = get_game_ids(20172018, 3)

print("Saison régulière :", len(regular_ids_2017))
print("Séries éliminatoires :", len(playoff_ids_2017))

In [47]:
# Tester le téléchargement d'un ancien match

old_game_id = regular_ids_2016[0]

old_game = download_game(old_game_id)

print("Game ID :", old_game["id"])
print("Date :", old_game["gameDate"])
print("Nombre d'événements :", len(old_game["plays"]))

Téléchargement depuis l'API : 2016020001
Game ID : 2016020001
Date : 2016-10-12
Nombre d'événements : 349


In [48]:
# Tester plusieurs saisons avec un seul match par type

seasons_test = [
    20162017,
    20192020,
    20202021,
    20232024,
]

for season_id in seasons_test:
    print("\n==============================")
    print("Saison :", season_id)
    print("==============================")

    download_season(season_id, max_games=1)


Saison : 20162017

Téléchargement de 1 matchs (regular) pour la saison 20162017
Téléchargement depuis l'API : 2016020001

Téléchargement de 1 matchs (playoffs) pour la saison 20162017
Téléchargement depuis l'API : 2016030111

Saison : 20192020

Téléchargement de 1 matchs (regular) pour la saison 20192020
Téléchargement depuis l'API : 2019020001

Téléchargement de 1 matchs (playoffs) pour la saison 20192020
Téléchargement depuis l'API : 2019030001

Saison : 20202021

Téléchargement de 1 matchs (regular) pour la saison 20202021
Téléchargement depuis l'API : 2020020001

Téléchargement de 1 matchs (playoffs) pour la saison 20202021
Téléchargement depuis l'API : 2020030111

Saison : 20232024

Téléchargement de 1 matchs (regular) pour la saison 20232024
Chargement depuis le cache : 2023020001.json

Téléchargement de 1 matchs (playoffs) pour la saison 20232024
Chargement depuis le cache : 2023030111.json


In [56]:
# Saisons demandées pour le projet
seasons = [
    20162017,
    20172018,
    20182019,
    20192020,
    20202021,
    20212022,
    20222023,
    20232024,
]

In [57]:
# Télécharger toutes les saisons demandées

for season_id in seasons:
    print("\n================================")
    print("Saison :", season_id)
    print("================================")

    # Premier téléchargement complet : saison 2016-2017
    download_season(20162017)


Saison : 20162017

Téléchargement de 1230 matchs (regular) pour la saison 20162017
Chargement depuis le cache : 2016020001.json
Téléchargement depuis l'API : 2016020002
Téléchargement depuis l'API : 2016020003
Téléchargement depuis l'API : 2016020004
Téléchargement depuis l'API : 2016020005
Téléchargement depuis l'API : 2016020006
Téléchargement depuis l'API : 2016020007
Téléchargement depuis l'API : 2016020008
Téléchargement depuis l'API : 2016020009
Téléchargement depuis l'API : 2016020010
Téléchargement depuis l'API : 2016020011
Téléchargement depuis l'API : 2016020012
Téléchargement depuis l'API : 2016020013
Téléchargement depuis l'API : 2016020014
Téléchargement depuis l'API : 2016020015
Téléchargement depuis l'API : 2016020016
Téléchargement depuis l'API : 2016020017
Téléchargement depuis l'API : 2016020018
Téléchargement depuis l'API : 2016020019
Téléchargement depuis l'API : 2016020020
Téléchargement depuis l'API : 2016020021
Téléchargement depuis l'API : 2016020022
Télécharge

In [59]:
# Télécharger la saison 2017-2018
download_season(20172018)


Téléchargement de 1271 matchs (regular) pour la saison 20172018
Chargement depuis le cache : 2017020001.json
Chargement depuis le cache : 2017020002.json
Chargement depuis le cache : 2017020003.json
Chargement depuis le cache : 2017020004.json
Chargement depuis le cache : 2017020005.json
Chargement depuis le cache : 2017020006.json
Chargement depuis le cache : 2017020007.json
Chargement depuis le cache : 2017020008.json
Chargement depuis le cache : 2017020009.json
Chargement depuis le cache : 2017020010.json
Chargement depuis le cache : 2017020011.json
Chargement depuis le cache : 2017020012.json
Chargement depuis le cache : 2017020013.json
Chargement depuis le cache : 2017020014.json
Chargement depuis le cache : 2017020015.json
Chargement depuis le cache : 2017020016.json
Chargement depuis le cache : 2017020017.json
Chargement depuis le cache : 2017020018.json
Chargement depuis le cache : 2017020019.json
Chargement depuis le cache : 2017020020.json
Chargement depuis le cache : 201702

In [60]:
# Vérifier le nombre attendu de matchs pour la saison 2017-2018

regular_ids_2017 = get_game_ids(20172018, 2)
playoff_ids_2017 = get_game_ids(20172018, 3)

print("Saison régulière :", len(regular_ids_2017))
print("Séries éliminatoires :", len(playoff_ids_2017))

Saison régulière : 1271
Séries éliminatoires : 84


In [61]:
# Télécharger la saison 2018-2019
download_season(20182019)


Téléchargement de 1271 matchs (regular) pour la saison 20182019
Téléchargement depuis l'API : 2018020001
Téléchargement depuis l'API : 2018020002
Téléchargement depuis l'API : 2018020003
Téléchargement depuis l'API : 2018020004
Téléchargement depuis l'API : 2018020005
Téléchargement depuis l'API : 2018020006
Téléchargement depuis l'API : 2018020007
Téléchargement depuis l'API : 2018020008
Téléchargement depuis l'API : 2018020009
Téléchargement depuis l'API : 2018020010
Téléchargement depuis l'API : 2018020011
Téléchargement depuis l'API : 2018020012
Téléchargement depuis l'API : 2018020013
Téléchargement depuis l'API : 2018020014
Téléchargement depuis l'API : 2018020015
Téléchargement depuis l'API : 2018020016
Téléchargement depuis l'API : 2018020017
Téléchargement depuis l'API : 2018020018
Téléchargement depuis l'API : 2018020019
Téléchargement depuis l'API : 2018020020
Téléchargement depuis l'API : 2018020021
Téléchargement depuis l'API : 2018020022
Téléchargement depuis l'API : 201

In [62]:
regular_ids_2018 = get_game_ids(20182019, 2)
playoff_ids_2018 = get_game_ids(20182019, 3)

print("Saison régulière :", len(regular_ids_2018))
print("Séries éliminatoires :", len(playoff_ids_2018))

Saison régulière : 1271
Séries éliminatoires : 87


In [63]:
# Télécharger la saison 2019-2020
download_season(20192020)


Téléchargement de 1082 matchs (regular) pour la saison 20192020
Chargement depuis le cache : 2019020001.json
Téléchargement depuis l'API : 2019020002
Téléchargement depuis l'API : 2019020003
Téléchargement depuis l'API : 2019020004
Téléchargement depuis l'API : 2019020005
Téléchargement depuis l'API : 2019020006
Téléchargement depuis l'API : 2019020007
Téléchargement depuis l'API : 2019020008
Téléchargement depuis l'API : 2019020009
Téléchargement depuis l'API : 2019020010
Téléchargement depuis l'API : 2019020011
Téléchargement depuis l'API : 2019020012
Téléchargement depuis l'API : 2019020013
Téléchargement depuis l'API : 2019020014
Téléchargement depuis l'API : 2019020015
Téléchargement depuis l'API : 2019020016
Téléchargement depuis l'API : 2019020017
Téléchargement depuis l'API : 2019020018
Téléchargement depuis l'API : 2019020019
Téléchargement depuis l'API : 2019020020
Téléchargement depuis l'API : 2019020021
Téléchargement depuis l'API : 2019020022
Téléchargement depuis l'API :

In [64]:
# Vérifier le nombre attendu de matchs pour la saison 2019-2020

regular_ids_2019 = get_game_ids(20192020, 2)
playoff_ids_2019 = get_game_ids(20192020, 3)

print("Saison régulière :", len(regular_ids_2019))
print("Séries éliminatoires :", len(playoff_ids_2019))

Saison régulière : 1082
Séries éliminatoires : 134


In [65]:
# Télécharger la saison 2020-2021
download_season(20202021)


Téléchargement de 868 matchs (regular) pour la saison 20202021
Chargement depuis le cache : 2020020001.json
Téléchargement depuis l'API : 2020020002
Téléchargement depuis l'API : 2020020003
Téléchargement depuis l'API : 2020020004
Téléchargement depuis l'API : 2020020005
Téléchargement depuis l'API : 2020020006
Téléchargement depuis l'API : 2020020007
Téléchargement depuis l'API : 2020020008
Téléchargement depuis l'API : 2020020009
Téléchargement depuis l'API : 2020020010
Téléchargement depuis l'API : 2020020011
Téléchargement depuis l'API : 2020020012
Téléchargement depuis l'API : 2020020013
Téléchargement depuis l'API : 2020020014
Téléchargement depuis l'API : 2020020015
Téléchargement depuis l'API : 2020020016
Téléchargement depuis l'API : 2020020017
Téléchargement depuis l'API : 2020020018
Téléchargement depuis l'API : 2020020019
Téléchargement depuis l'API : 2020020020
Téléchargement depuis l'API : 2020020021
Téléchargement depuis l'API : 2020020022
Téléchargement depuis l'API : 

In [66]:
# Vérifier le nombre attendu de matchs pour la saison 2020-2021

regular_ids_2020 = get_game_ids(20202021, 2)
playoff_ids_2020 = get_game_ids(20202021, 3)

print("Saison régulière :", len(regular_ids_2020))
print("Séries éliminatoires :", len(playoff_ids_2020))

Saison régulière : 868
Séries éliminatoires : 105


In [67]:
# Télécharger la saison 2021-2022
download_season(20212022)


Téléchargement de 1312 matchs (regular) pour la saison 20212022
Téléchargement depuis l'API : 2021020001
Téléchargement depuis l'API : 2021020002
Téléchargement depuis l'API : 2021020003
Téléchargement depuis l'API : 2021020004
Téléchargement depuis l'API : 2021020005
Téléchargement depuis l'API : 2021020006
Téléchargement depuis l'API : 2021020007
Téléchargement depuis l'API : 2021020008
Téléchargement depuis l'API : 2021020009
Téléchargement depuis l'API : 2021020010
Téléchargement depuis l'API : 2021020011
Téléchargement depuis l'API : 2021020012
Téléchargement depuis l'API : 2021020013
Téléchargement depuis l'API : 2021020014
Téléchargement depuis l'API : 2021020015
Téléchargement depuis l'API : 2021020016
Téléchargement depuis l'API : 2021020017
Téléchargement depuis l'API : 2021020018
Téléchargement depuis l'API : 2021020019
Téléchargement depuis l'API : 2021020020
Téléchargement depuis l'API : 2021020021
Téléchargement depuis l'API : 2021020022
Téléchargement depuis l'API : 202

In [68]:
regular_ids_2021 = get_game_ids(20212022, 2)
playoff_ids_2021 = get_game_ids(20212022, 3)

print("Saison régulière :", len(regular_ids_2021))
print("Séries éliminatoires :", len(playoff_ids_2021))

Saison régulière : 1312
Séries éliminatoires : 105


In [69]:
# Télécharger la saison 2022-2023
download_season(20222023)


Téléchargement de 1312 matchs (regular) pour la saison 20222023
Téléchargement depuis l'API : 2022020001
Téléchargement depuis l'API : 2022020002
Téléchargement depuis l'API : 2022020003
Téléchargement depuis l'API : 2022020004
Téléchargement depuis l'API : 2022020005
Téléchargement depuis l'API : 2022020006
Téléchargement depuis l'API : 2022020007
Téléchargement depuis l'API : 2022020008
Téléchargement depuis l'API : 2022020009
Téléchargement depuis l'API : 2022020010
Téléchargement depuis l'API : 2022020011
Téléchargement depuis l'API : 2022020012
Téléchargement depuis l'API : 2022020013
Téléchargement depuis l'API : 2022020014
Téléchargement depuis l'API : 2022020015
Téléchargement depuis l'API : 2022020016
Téléchargement depuis l'API : 2022020017
Téléchargement depuis l'API : 2022020018
Téléchargement depuis l'API : 2022020019
Téléchargement depuis l'API : 2022020020
Téléchargement depuis l'API : 2022020021
Téléchargement depuis l'API : 2022020022
Téléchargement depuis l'API : 202

In [70]:
regular_ids_2022 = get_game_ids(20222023, 2)
playoff_ids_2022 = get_game_ids(20222023, 3)

print("Saison régulière :", len(regular_ids_2022))
print("Séries éliminatoires :", len(playoff_ids_2022))

Saison régulière : 1312
Séries éliminatoires : 105


In [71]:
# Télécharger la saison 2023-2024
download_season(20232024)


Téléchargement de 1312 matchs (regular) pour la saison 20232024
Chargement depuis le cache : 2023020001.json
Chargement depuis le cache : 2023020002.json
Téléchargement depuis l'API : 2023020003
Téléchargement depuis l'API : 2023020004
Téléchargement depuis l'API : 2023020005
Téléchargement depuis l'API : 2023020006
Téléchargement depuis l'API : 2023020007
Téléchargement depuis l'API : 2023020008
Téléchargement depuis l'API : 2023020009
Téléchargement depuis l'API : 2023020010
Téléchargement depuis l'API : 2023020011
Téléchargement depuis l'API : 2023020012
Téléchargement depuis l'API : 2023020013
Téléchargement depuis l'API : 2023020014
Téléchargement depuis l'API : 2023020015
Téléchargement depuis l'API : 2023020016
Téléchargement depuis l'API : 2023020017
Téléchargement depuis l'API : 2023020018
Téléchargement depuis l'API : 2023020019
Téléchargement depuis l'API : 2023020020
Téléchargement depuis l'API : 2023020021
Téléchargement depuis l'API : 2023020022
Téléchargement depuis l'A

In [72]:
regular_ids_2023 = get_game_ids(20232024, 2)
playoff_ids_2023 = get_game_ids(20232024, 3)

print("Saison régulière :", len(regular_ids_2023))
print("Séries éliminatoires :", len(playoff_ids_2023))

Saison régulière : 1312
Séries éliminatoires : 105


In [73]:
from pathlib import Path

seasons = [
    20162017,
    20172018,
    20182019,
    20192020,
    20202021,
    20212022,
    20222023,
    20232024,
]

data_root = Path("../data/raw")

for season_id in seasons:
    expected_regular = len(get_game_ids(season_id, 2))
    expected_playoffs = len(get_game_ids(season_id, 3))

    regular_dir = data_root / str(season_id) / "regular"
    playoffs_dir = data_root / str(season_id) / "playoffs"

    local_regular = len(list(regular_dir.glob("*.json")))
    local_playoffs = len(list(playoffs_dir.glob("*.json")))

    print(f"Saison {season_id}")
    print(
        f"  Regular  : {local_regular}/{expected_regular}"
    )
    print(
        f"  Playoffs : {local_playoffs}/{expected_playoffs} "
        f"(sans PBP : {expected_playoffs - local_playoffs})"
    )
    print()

Saison 20162017
  Regular  : 1230/1230
  Playoffs : 87/87 (sans PBP : 0)

Saison 20172018
  Regular  : 1271/1271
  Playoffs : 84/84 (sans PBP : 0)

Saison 20182019
  Regular  : 1271/1271
  Playoffs : 87/87 (sans PBP : 0)

Saison 20192020
  Regular  : 1082/1082
  Playoffs : 130/134 (sans PBP : 4)

Saison 20202021
  Regular  : 868/868
  Playoffs : 84/105 (sans PBP : 21)

Saison 20212022
  Regular  : 1312/1312
  Playoffs : 89/105 (sans PBP : 16)

Saison 20222023
  Regular  : 1312/1312
  Playoffs : 88/105 (sans PBP : 17)

Saison 20232024
  Regular  : 1312/1312
  Playoffs : 88/105 (sans PBP : 17)

